In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-6A: GRPO and its refinements, appendix

## Where this session fits

> **The limitation we start from: RLLM-6 gave the core of each change in a line or two. This optional appendix holds the detail and the evidence it left out.**

- **The question:** what is the rest of the argument for each change, and how far does the evidence go?
- **Position:** an appendix to RLLM-6 (GRPO and its refinements): RLLM-7 does not need it
    - read it after RLLM-6, or look up one slide when RLLM-6 points here
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. give Dr. GRPO's argument that dividing by length makes wrong answers drift longer
2. write GSPO's completion ratio and say why its clip ranges are so small
3. say what dropping the KL term risks, and whose reading that is
4. say what the DAPO ablation shows, and what it does not
5. group the changes by the kind of evidence behind them


## Averaging: Dr. GRPO's argument

> **Dr. GRPO (its argument): with a negative advantage, dividing by the completion's length punishes a long wrong answer less per token.**

- advantage $-1$: share 0.05 per token for 10 tokens, 0.005 for 100 (RLLM-6's group of two)
- so wrong answers drift longer: Dr. GRPO divides by a constant (the generation budget) instead
- **our reading:** the length growth seen in R1-Zero (the model RL-trained straight from a base model, RLLM-5) may come partly from this loss artifact
- **the paper's:** reflection is already present in base models, before RL
    - this is our pointer only: these decks do not examine reflection


## Clip: GSPO's completion ratio in detail

> **The completion ratio is the geometric mean of the token ratios, and its clip range is tiny because a mean over many tokens moves little.**

- $s_i(\theta) = \exp\bigl(\frac{1}{\lvert \y_i \rvert} \sum_{\tt} \log r_{i,\tt}(\theta)\bigr)$
    - the figure in RLLM-6 draws the clip range $[0.8, 1.2]$ for visibility
- GSPO's real clip ranges are $3 \times 10^{-4}$ and $4 \times 10^{-4}$: not a typo, since the ratio is a mean over tokens
    - over $1000$ tokens, a mean log ratio of $4 \times 10^{-4}$ is a total of $0.4$ (our arithmetic)
- the paper reports more stable training, and a benefit for **mixture-of-experts (MoE)** models
    - an MoE model routes each token to a few expert sub-networks
    - after an update the routing can change, so token ratios swing
    - **routing replay**, the paper's workaround: reuse the old router choices. GSPO makes it unnecessary
- the evidence is from the paper's own Qwen models


## KL term: what follows from dropping it, and whose reading it is

> **With no KL, only the reward and the clip hold the policy, and the clip is a heuristic brake (RLLM-4).**

- with a learned reward model the KL keeps the policy where that model is accurate (RLLM-2)
- **our reading, not a claim of either paper:** drop it with a verifier, think twice with a learned reward model
- the benefit: no reference model in memory
- the risk: drift that the reward does not forbid
- with $\beta = 0$ you can still log the KL to the reference (RLLM-5's signals table), but only if you keep the reference model


## Evidence: what the DAPO ablation shows, and does not

> **Each change was added on top of the previous ones; the score rose at every step, from 30 to 50.**

<img src="images/rllm6_ablation.png" width=50%>

- the paper's numbers, cumulative in the authors' order: a step's gain depends on the steps before it
- token-level loss is RLLM-6's Averaging, clip higher its Clip, dynamic sampling its Group filtering, the two overlong steps its Cut-offs
- Dr. GRPO's std, the KL drop and GSPO's ratio have no number here
- one base model, one benchmark, the authors' own runs: "these changes helped here", not a ranking


## Evidence: the seven changes by kind of evidence

> **Group the changes by the kind of evidence behind them. These are options, not a ranking.**

- **backed by arithmetic:** token-level averaging weights (Averaging), all-equal groups wasted (Group filtering)
- **backed by one paper's ablation:** clip higher (Clip), dynamic sampling (Group filtering), overlong handling (Cut-offs)
- **argument or design choice only:** dropping the std division (Advantage scale), dropping the KL (KL term), GSPO's completion ratio (Clip)
- one base model, one benchmark: "helped there", not "always helps"

**Papers (arXiv):** DAPO 2503.14476; Dr. GRPO, "Understanding R1-Zero-Like Training", 2503.20783; GSPO 2507.18071


# What we established, and what it sets up

- **Established**
    - Dr. GRPO: dividing by length lets wrong answers grow longer, so divide by a constant
    - GSPO: one ratio per completion, with a tiny clip range because it is a mean
    - dropping the KL is a design choice: our reading is to drop it with a verifier only
    - the DAPO ablation is cumulative, one model, one benchmark: evidence of "helped there"
- **Sets up**
    - **RLLM-7, the start of part 2 (RFT Alternative methods):** varying the three axes, as RLLM-6 closed
    - **RLLM-13:** the full decision rule, with the Part 2 options
